# Instruction fine-tuning with Accelerate

This notebook follows `model_training/pytorch/01_test_multi_gpu.ipynb`: define training in a function, then run that function with `notebook_launcher`.

> Restart this kernel before the launch cell. Do not run CUDA device checks, load a model, or move tensors to CUDA in the notebook parent process. Each Accelerate worker creates its own data loader, model, optimizer, and CUDA state.

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import importlib.util
import logging
import sys
import time

import torch
from accelerate import Accelerator, notebook_launcher

project_root = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / 'model_training').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Run this notebook from inside the deep_learning repository')

sys.path.insert(0, str(project_root))
from model_training.framework.data.post_train.instruction_tuning.interfaces import (
    create_sft_dataloaders,
    load_instruction_records,
    prepare_sft_data,
)
from model_training.framework.model.load_model.gpt2.model import return_model
from model_training.framework.train.core.config import TrainConfig
from model_training.framework.train.loss.sft import sft_forward, sft_loss
from model_training.framework.train.utils.lr_scheduler import cosine_lr
from model_training.framework.train.utils.optimizer import configure_optimizer

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s | %(message)s')

In [ ]:
SFT_CONFIG = {
    'data_path': str(project_root / 'datasets/instruction_tuning/instruction-data.json'),
    'output_dir': str(project_root / 'execution_results/qwen_instruction'),
    'model_name': 'Qwen/Qwen2.5-3B-Instruct',
    'num_processes': 2,
    'mixed_precision': 'fp16',
    'epochs': 2,
    'batch_size': 1,
    'accumulation_steps': 4,
    'max_length': 1024,
    'learning_rate': 1e-5,
    'max_lr': 1e-5,
    'min_lr': 1e-5,
    'warmup_steps': 0,
    'weight_decay': 0.1,
    'grad_clip': 1.0,
    'train_fraction': 0.85,
    'test_fraction': 0.10,
    'num_workers': 0,
}

print('[STAGE] configuration')
print(f"✓ model: {SFT_CONFIG['model_name']}")
print(f"✓ processes: {SFT_CONFIG['num_processes']} | precision: {SFT_CONFIG['mixed_precision']}")
print(f"✓ batch/GPU: {SFT_CONFIG['batch_size']} | accumulation: {SFT_CONFIG['accumulation_steps']}")
print(f"✓ output: {SFT_CONFIG['output_dir']}")

## 2. Validate the launch inputs

This check deliberately does not call `torch.cuda.*`; CUDA must first initialize inside the workers.

In [ ]:
print('[STAGE] validate launch inputs')
missing = [name for name in ('torch', 'transformers', 'accelerate') if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(f"Missing packages in this kernel: {', '.join(missing)}")

data_path = Path(SFT_CONFIG['data_path'])
if not data_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

records = load_instruction_records(data_path)
if SFT_CONFIG['num_processes'] < 1:
    raise ValueError('num_processes must be at least one')
print(f"✓ records: {len(records)} | training split: {int(len(records) * SFT_CONFIG['train_fraction'])}")
print('✓ CUDA is still uninitialized in this notebook parent process')

## 3. Define the framework-SFT worker functions

The functions below preserve the original SFT pipeline's framework model loader, `TrainConfig`, optimizer grouping, cosine schedule, SFT forward step, and SFT loss. Accelerate only takes over distributed preparation, backward propagation, and metric aggregation.

In [ ]:
def build_sft_training_components(config: dict[str, object], accelerator: Accelerator):
    """Build the same SFT data, model, optimizer, and TrainConfig as the lesson."""
    records = load_instruction_records(Path(str(config['data_path'])))
    prepared_sft = prepare_sft_data(
        records,
        model_name=str(config['model_name']),
        train_fraction=float(config['train_fraction']),
        test_fraction=float(config['test_fraction']),
    )
    loaders = create_sft_dataloaders(
        prepared_sft,
        batch_size=int(config['batch_size']),
        max_length=int(config['max_length']),
        num_workers=int(config['num_workers']),
    )

    model = return_model(str(config['model_name']))
    model.config.use_cache = False
    optimizer = configure_optimizer(
        model,
        weight_decay=float(config['weight_decay']),
        learning_rate=float(config['learning_rate']),
        device=accelerator.device.type,
        master_process=accelerator.is_main_process,
    )
    model, optimizer, train_loader = accelerator.prepare(model, optimizer, loaders.train)

    accumulation_steps = int(config['accumulation_steps'])
    num_steps = (int(config['epochs']) * len(train_loader)) // accumulation_steps
    if num_steps < 1:
        raise ValueError('The training loader is too short for the configured accumulation steps')
    train_config = TrainConfig(
        num_steps=num_steps,
        learning_rate=float(config['learning_rate']),
        device=accelerator.device,
        total_batch_size=(
            int(config['batch_size'])
            * int(config['max_length'])
            * accumulation_steps
            * accelerator.num_processes
        ),
        micro_batch_size=int(config['batch_size']),
        seq_length=int(config['max_length']),
        accumulation_steps=accumulation_steps,
        max_lr=float(config['max_lr']),
        min_lr=float(config['min_lr']),
        warmup_steps=int(config['warmup_steps']),
        weight_decay=float(config['weight_decay']),
        grad_clip=float(config['grad_clip']),
        use_compile=False,
        precision=str(config['mixed_precision']),
    )
    return prepared_sft, model, optimizer, train_loader, train_config


def next_sft_batch(train_iterator, train_loader):
    """Return the next batch, restarting the prepared loader when an epoch ends."""
    try:
        return next(train_iterator), train_iterator
    except StopIteration:
        train_iterator = iter(train_loader)
        return next(train_iterator), train_iterator


def run_accelerated_sft_steps(model, optimizer, train_loader, train_config, accelerator) -> None:
    """Run the lesson's SFT forward, loss, accumulation, clipping, and cosine schedule."""
    train_iterator = iter(train_loader)
    model.train()

    for step in range(train_config.num_steps):
        accelerator.wait_for_everyone()
        start_time = time.perf_counter()
        optimizer.zero_grad()
        loss_accum = torch.zeros((), device=accelerator.device)

        for _ in range(train_config.accumulation_steps):
            batch, train_iterator = next_sft_batch(train_iterator, train_loader)
            with accelerator.autocast():
                logits, labels = sft_forward(model, batch, accelerator.device)
                loss = sft_loss(logits, labels)
            accelerator.backward(loss / train_config.accumulation_steps)
            loss_accum += loss.detach() / train_config.accumulation_steps

        grad_norm = accelerator.clip_grad_norm_(model.parameters(), train_config.grad_clip)
        learning_rate = cosine_lr(
            step=step,
            total_steps=train_config.num_steps,
            warmup_steps=train_config.warmup_steps,
            peak_lr=train_config.max_lr,
            min_lr=train_config.min_lr,
        )
        for parameter_group in optimizer.param_groups:
            parameter_group['lr'] = learning_rate
        optimizer.step()

        accelerator.wait_for_everyone()
        elapsed_seconds = time.perf_counter() - start_time
        global_loss = accelerator.reduce(loss_accum, reduction='mean')
        global_norm = accelerator.reduce(grad_norm.detach(), reduction='mean')
        if accelerator.is_main_process:
            tokens_per_second = (
                train_config.total_batch_size / elapsed_seconds
            )
            print(
                f"step {step} | loss: {global_loss.item():.6f} | "
                f"norm: {global_norm.item():.2f} | "
                f"dt: {elapsed_seconds * 1000:.2f} ms | "
                f"tok/sec: {tokens_per_second:.2f}",
                flush=True,
            )


def save_sft_artifacts(prepared_sft, model, accelerator, output_dir: Path) -> None:
    """Save the unwrapped model and tokenizer once after all workers finish."""
    if accelerator.is_main_process:
        output_dir.mkdir(parents=True, exist_ok=True)
    accelerator.wait_for_everyone()

    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(
        output_dir,
        is_main_process=accelerator.is_main_process,
        save_function=accelerator.save,
        state_dict=accelerator.get_state_dict(model),
    )
    if accelerator.is_main_process:
        prepared_sft.tokenizer.save_pretrained(output_dir)
        print(f"✓ saved model and tokenizer: {output_dir}", flush=True)
    accelerator.wait_for_everyone()


def train_instruction_sft(config: dict[str, object]) -> None:
    """Run the original notebook's SFT training stack in one Accelerate worker."""
    accelerator = Accelerator(mixed_precision=str(config['mixed_precision']))
    prepared_sft, model, optimizer, train_loader, train_config = build_sft_training_components(
        config, accelerator
    )
    print(
        f"[WORKER {accelerator.process_index}] device={accelerator.device} "
        f"batches={len(train_loader)} | steps={train_config.num_steps}",
        flush=True,
    )
    run_accelerated_sft_steps(
        model, optimizer, train_loader, train_config, accelerator
    )
    save_sft_artifacts(
        prepared_sft, model, accelerator, Path(str(config['output_dir']))
    )

## 4. Launch all workers

Run this only after restarting the kernel and executing the preceding cells in order.

In [ ]:
print('[STAGE] launch Accelerate workers', flush=True)
notebook_launcher(
    train_instruction_sft,
    args=(SFT_CONFIG,),
    num_processes=int(SFT_CONFIG['num_processes']),
    mixed_precision=str(SFT_CONFIG['mixed_precision']),
)
print('✓ all workers completed', flush=True)

## 5. Inspect saved artifacts

In [ ]:
output_dir = Path(SFT_CONFIG['output_dir'])
print(f"[STAGE] inspect saved artifacts | {output_dir}")
if not output_dir.is_dir():
    raise FileNotFoundError(f"Output directory not found: {output_dir}")
artifacts = sorted(path.name for path in output_dir.iterdir() if path.is_file())
print('✓ files:', artifacts)